## Read the Bronze table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

categories = spark.table(
    "e2e_project.bronze.erp_px_cat_g1v2"
)

display(categories)

In [0]:
categories.printSchema()
print("Rows:", categories.count())

## Inspect the raw categories first

In [0]:
display(categories)

In [0]:
display(
    categories
    .groupBy("CAT")
    .count()
    .orderBy("CAT")
)

In [0]:
display(
    categories
    .groupBy("SUBCAT")
    .count()
    .orderBy("SUBCAT")
)

Also inspect maintenance values:

In [0]:
display(
    categories
    .groupBy("MAINTENANCE")
    .count()
)

## Trim all string values

In [0]:
for field in categories.schema.fields:
    if isinstance(field.dataType, StringType):
        categories = categories.withColumn(
            field.name,
            trim(col(field.name))
        )

## Check the category IDs

In [0]:
display(
    categories.select("ID").orderBy("ID")
)

CRM products.cat_id
        =
ERP categories.ID

let's check whether the two systems actually match.

In [0]:
crm_category_ids = (
    spark.table("e2e_project.silver.crm_prd_info")
    .select("cat_id")
    .distinct()
)

erp_category_ids = (
    categories
    .select(
        col("ID").alias("cat_id")
    )
    .distinct()
)

Now find CRM category IDs that don't exist in ERP:

In [0]:
unmatched_categories = (
    crm_category_ids
    .join(
        erp_category_ids,
        on="cat_id",
        how="left_anti"
    )
)

display(unmatched_categories)

CRM category IDs

        │
        │ compare

        ▼

ERP category IDs


left_anti

        ↓

CRM IDs with no ERP match


This is extremely useful for data quality and integration testing.

Ideally the result is empty or contains only values we can explain.

## Check duplicate ERP category IDs

In [0]:
display(
    categories
    .groupBy("ID")
    .count()
    .filter(col("count") > 1)
)

In [0]:
display(
    categories
    .groupBy("ID", "CAT", "SUBCAT")
    .count()
    .filter(col("count") > 1)
)


## Check NULL or blank category information

In [0]:
display(
    categories.filter(
        col("ID").isNull() |
        (col("ID") == "") |
        col("CAT").isNull() |
        (col("CAT") == "") |
        col("SUBCAT").isNull() |
        (col("SUBCAT") == "")
    )
)

In [0]:
display(
    categories.filter(
        col("ID").isNull() |
        (col("ID") == "") |
        col("CAT").isNull() |
        (col("CAT") == "") |
        col("SUBCAT").isNull() |
        (col("SUBCAT") == "")
    )
)

## Standardize missing maintenance values

In [0]:
categories = categories.withColumn(
    "MAINTENANCE",
    F.when(
        col("MAINTENANCE").isNull() |
        (col("MAINTENANCE") == ""),
        "n/a"
    )
    .otherwise(col("MAINTENANCE"))
)

## Rename the columns

In [0]:
RENAME_MAP = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE": "maintenance"
}

for old_name, new_name in RENAME_MAP.items():
    categories = categories.withColumnRenamed(
        old_name,
        new_name
    )

## Inspect the final table

In [0]:
display(categories)

In [0]:
categories.printSchema()

This table is small, but it's very important because it provides the business hierarchy missing from CRM product data.

## Validate unwanted whitespace

In [0]:
display(
    categories.filter(
        (col("category") != trim(col("category"))) |
        (col("subcategory") != trim(col("subcategory"))) |
        (col("maintenance") != trim(col("maintenance")))
    )
)

## Save the final Silver table

In [0]:
(
    categories.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "e2e_project.silver.erp_product_category"
    )
)

In [0]:
%sql

SELECT *
FROM e2e_project.silver.erp_product_category;

Silver is now complete